In [11]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import inequality_analyzers as inqA
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise

In [12]:
# Load cleaned data
df = pd.read_csv(data_root / 'CBOS_data.csv')

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_95626/246268884.py:2: DtypeWarning: Columns (14,15,20,26,27,28,32,33,34,39,40,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_root / 'CBOS_data.csv')


In [13]:
# Get rid of invalid sex
df['sex'] = df['sex'].apply(lambda x: np.nan if x not in [1.0, 2.0] else x)

In [14]:
# Handle age
df['age'] = np.nan
for id, row in df.iterrows():
    age = row['survey_year'] - row['year_born']
    if age < 0:
        print(f"Warning: Negative age for id {id} (survey year: {row['survey_year']}, survey month: {row['survey_month']}, year born: {row['year_born']})")
    elif age > 200:
        yb = row['year_born']
        if len(str(yb)) == 3:
            yb = '0' + str(yb)
        else:
            yb = str(yb)    
        
        full_year = str('19' + yb)
        
        if len(full_year) == 6:
            full_year = full_year[:-2]
        
        intyear = int(full_year)
        df.at[id, 'age'] = row['survey_year'] - intyear
    else:
        df.at[id, 'age'] = age

In [15]:
# Unique pairs sex and sex_L
display(df[['sex', 'sex_L']].drop_duplicates())
SEX_MAPPING = {1.0: 'Mężczyzna', 2.0: 'Kobieta'}
df['sex_L'] = df['sex'].map(SEX_MAPPING)
display(df[['sex', 'sex_L']].drop_duplicates())

,sex,sex_L
0,2.0,Kobieta
3,1.0,Mężczyzna
1184,NaN,NaN
10318,1.0,Mężczyźni
10320,2.0,Kobiety
30689,NaN,Brak danych / Odmowa odpowiedzi
32867,NaN,BRAK DANYCH / Odmowa odpowiedzi
36311,NaN,Brak danych/ Odmowa odpowiedzi
219430,2.0,Kobieta
219431,1.0,Mężczyzna


,sex,sex_L
0,2.0,Kobieta
3,1.0,Mężczyzna
1184,NaN,NaN


In [16]:
# Location variable
df['location_old'] = df['location_old'].apply(lambda x: np.nan if x not in [float(x) for x in range(1,50)] else x)
for idx, row in df.iterrows():
    if np.isnan(row['location_old']):
        df.at[idx, 'location_old_L'] = np.nan
    if row['location_old'] == 2.0:
        df.at[idx, 'location_old_L'] = 'bialskopodlaskie'
df['location_old_L'] = df['location_old_L'].str.lower()

display(df[['location_old', 'location_old_L']].drop_duplicates())

,location_old,location_old_L
0,1.0,warszawskie
2,5.0,bydgoskie
4,NaN,NaN
7,3.0,białostockie
8,2.0,bialskopodlaskie
11,4.0,bielskie
1486,29.0,pilskie
1504,31.0,płockie
1525,38.0,skierniewickie
1558,40.0,suwalskie


In [17]:
# Location variable
df['location_new'] = df['location_new'].apply(lambda x: np.nan if x not in [float(x) for x in range(1,50)] else x)
for idx, row in df.iterrows():
    if np.isnan(row['location_new']):
        df.at[idx, 'location_new_L'] = np.nan
df['location_new_L'] = df['location_new_L'].str.lower()
df['location_new_L'] = df['location_new_L'].str.strip()

display(df[['location_new', 'location_new_L']].drop_duplicates())

,location_new,location_new_L
0,NaN,NaN
122965,6.0,małopolskie
123005,12.0,śląskie
123032,11.0,pomorskie
123046,16.0,zachodniopomorskie
123139,7.0,mazowieckie
123167,5.0,łódzkie
123244,3.0,lubelskie
123373,2.0,kujawsko-pomorskie
123394,15.0,wielkopolskie


In [18]:
# City size
y = pd.DataFrame()
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    df_sub['city_size_L'] = df_sub['city_size_L'].str.strip()
    df_sub['city_size_L'] = df_sub['city_size_L'].str.lower()
    x = df_sub[['city_size', 'city_size_L']].drop_duplicates().sort_values(by='city_size').reset_index(drop=True)
    
    if x.equals(y):
        print(f"File {file} has consistent city_size and city_size_L values.")
    else:
        x = pd.DataFrame(x)
        #display(x)
    y = x
    

File CBOS_2_02_1990.sav has consistent city_size and city_size_L values.
File CBOS_4_04_1990.sav has consistent city_size and city_size_L values.
File CBOS_5_05_1990.sav has consistent city_size and city_size_L values.
File CBOS_6_06_1990.sav has consistent city_size and city_size_L values.
File CBOS_8_09_1990.sav has consistent city_size and city_size_L values.
File CBOS_9_10_1990.sav has consistent city_size and city_size_L values.
File CBOS_10_11_1990.sav has consistent city_size and city_size_L values.
File CBOS_11_12_1990.sav has consistent city_size and city_size_L values.
File CBOS_13_02_1991.sav has consistent city_size and city_size_L values.
File CBOS_14_03_1991.sav has consistent city_size and city_size_L values.
File CBOS_15_04_1991.sav has consistent city_size and city_size_L values.
File CBOS_16_05_1991.sav has consistent city_size and city_size_L values.
File CBOS_17_06_1991.sav has consistent city_size and city_size_L values.
File CBOS_20_09_1991.sav has consistent city

In [ ]:
CITY_SIZE_MAPPING = {
    1.0: 'wieś',
    2.0: 'do 20 tys. mieszkańców',
    3.0: '20-100 tys. mieszkańców',
    4.0: '100-500 tys. mieszkańców',
    5.0: 'powyżej 500 tys. mieszkańców'
}
KLM6_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0],
    3.0: [3.0, 4.0],
    4.0: [5.0],
    5.0: [6.0],
    np.nan: [8.0]
}
KLM7_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0],
    3.0: [3.0, 4.0],
    4.0: [5.0, 6.0],
    5.0: [7.0]
}
KLM9_NEW_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0, 3.0, 4.0, 5.0],
    3.0: [6.0, 7.0],
    4.0: [8.0],
    5.0: [9.0]
}
KLM9_OLD_CITY_SIZE_MAPPING = {
    1.0: [1.0],
    2.0: [2.0, 3.0, 4.0],
    3.0: [5.0, 6.0],
    4.0: [7.0, 8.0],
    5.0: [9.0]
}

# CBOS_49_05_1994.sav has a different city size variable. It is not possible to map it to the same city size variable as in other files, so we will keep it as is.

In [22]:
# City size
y = pd.DataFrame()
for file in df['survey_file'].unique():
    df_sub = df[df['survey_file'] == file].copy()
    df_sub['education_L'] = df_sub['education_L'].str.strip()
    df_sub['education_L'] = df_sub['education_L'].str.lower()
    x = df_sub[['education', 'education_L']].drop_duplicates().sort_values(by='education').reset_index(drop=True)
    for idx, row in x.iterrows():
        # Drop the rows that have np.nan in both education and education_L
        if pd.isna(row['education']) and pd.isna(row['education_L']):
            x = x.drop(idx)
    
    if x.equals(y):
        print(f"File {file} has consistent education and education_L values.")
    else:
        x = pd.DataFrame(x)
        display(x)
    y = x
    

,education,education_L
0,1.0,niepełne podstawowe
1,2.0,podstawowe ukończone
2,3.0,zasadnicze zawodowe
3,4.0,niepełne średnie
4,5.0,średnie ogólnokształcące
5,6.0,średnie zawodowe
6,7.0,pomaturalne
7,8.0,niepełne wyższe
8,9.0,wyższe


File CBOS_2_02_1990.sav has consistent education and education_L values.
File CBOS_3_03_1990.sav has consistent education and education_L values.
File CBOS_4_04_1990.sav has consistent education and education_L values.
File CBOS_5_05_1990.sav has consistent education and education_L values.
File CBOS_6_06_1990.sav has consistent education and education_L values.
File CBOS_7_07_1990.sav has consistent education and education_L values.


,education,education_L
0,1.0,niepełne podstawowe
1,2.0,podstawowe
2,3.0,zasadnicze zawodowe
3,4.0,niepełne średnie
4,5.0,średnie ogólnokształcące
5,6.0,średnie zawodowe
6,7.0,pomaturalne
7,8.0,niepełne wyższe
8,9.0,wyższe


File CBOS_9_10_1990.sav has consistent education and education_L values.
File CBOS_10_11_1990.sav has consistent education and education_L values.
File CBOS_11_12_1990.sav has consistent education and education_L values.
File CBOS_12_01_1991.sav has consistent education and education_L values.
File CBOS_13_02_1991.sav has consistent education and education_L values.
File CBOS_14_03_1991.sav has consistent education and education_L values.
File CBOS_15_04_1991.sav has consistent education and education_L values.
File CBOS_16_05_1991.sav has consistent education and education_L values.
File CBOS_17_06_1991.sav has consistent education and education_L values.
File CBOS_18_07_1991.sav has consistent education and education_L values.
File CBOS_19_08_1991.sav has consistent education and education_L values.
File CBOS_20_09_1991.sav has consistent education and education_L values.
File CBOS_21_10_1991.sav has consistent education and education_L values.
File CBOS_22_11_1991.sav has consistent

,education,education_L
0,1.0,niepełne podstawowe
1,2.0,podstawowe
2,3.0,zasadnicze
3,4.0,nepełne średnie
4,5.0,średnie ogólnokształcące
5,6.0,średnie zawodowe
6,7.0,pomaturalne
7,8.0,niepełne wyższe
8,9.0,wyższe


...

,education,education_L
0,1.0,nieukończone podstawowe lub bez wykształcenia ...
1,2.0,podstawowe
2,3.0,gimnazjalne
3,4.0,zasadnicze zawodowe (także spr)
4,5.0,średnie ogólnokształcące bez matury
5,6.0,średnie ogólnokształcące z maturą
6,7.0,średnie zawodowe bez matury
7,8.0,"średnie zawodowe z maturą (technikum, liceum z..."
8,9.0,pomaturalne lub policealne
9,10.0,"wyższe z tytułem inżyniera, licencjata, dyplom..."


File CBOS_302_07_2015.sav has consistent education and education_L values.
File CBOS_303_08_2015.sav has consistent education and education_L values.
File CBOS_304_09_2015.sav has consistent education and education_L values.
File CBOS_305_10_2015.sav has consistent education and education_L values.
File CBOS_306_11_2015.sav has consistent education and education_L values.
File CBOS_307_12_2015.sav has consistent education and education_L values.
File CBOS_308_01_2016.sav has consistent education and education_L values.
File CBOS_309_02_2016.sav has consistent education and education_L values.
File CBOS_310_03_2016.sav has consistent education and education_L values.
File CBOS_311_04_2016.sav has consistent education and education_L values.
File CBOS_312_05_2016.sav has consistent education and education_L values.
File CBOS_313_06_2016.sav has consistent education and education_L values.
File CBOS_314_07_2016.sav has consistent education and education_L values.
File CBOS_315_08_2016.sav

,education,education_L
0,2.0,podstawowe
1,3.0,gimnazjalne
2,4.0,zasadnicze zawodowe (także spr)
3,5.0,średnie ogólnokształcące bez matury
4,6.0,średnie ogólnokształcące z maturą
5,7.0,średnie zawodowe bez matury
6,8.0,"średnie zawodowe z maturą (technikum, liceum z..."
7,9.0,pomaturalne lub policealne
8,10.0,"wyższe z tytułem inżyniera, licencjata, dyplom..."
9,11.0,"wyższe z tytułem magistra, lekarza lub równorz..."


,education,education_L
0,1.0,nieukończone podstawowe lub bez wykształcenia ...
1,2.0,podstawowe
2,3.0,gimnazjalne
3,4.0,zasadnicze zawodowe (także spr)
4,5.0,średnie ogólnokształcące bez matury
5,6.0,średnie ogólnokształcące z maturą
6,7.0,średnie zawodowe bez matury
7,8.0,"średnie zawodowe z maturą (technikum, liceum z..."
8,9.0,pomaturalne lub policealne
9,10.0,"wyższe z tytułem inżyniera, licencjata, dyplom..."


File CBOS_319_12_2016.sav has consistent education and education_L values.
File CBOS_320_01_2017.sav has consistent education and education_L values.
File CBOS_321_02_2017.sav has consistent education and education_L values.
File CBOS_322_03_2017.sav has consistent education and education_L values.
File CBOS_323_04_2017.sav has consistent education and education_L values.
File CBOS_324_05_2017.sav has consistent education and education_L values.
File CBOS_325_06_2017.sav has consistent education and education_L values.
File CBOS_326_07_2017.sav has consistent education and education_L values.
File CBOS_327_08_2017.sav has consistent education and education_L values.
File CBOS_328_09_2017.sav has consistent education and education_L values.
File CBOS_329_10_2017.sav has consistent education and education_L values.
File CBOS_330_11_2017.sav has consistent education and education_L values.
File CBOS_331_12_2017.sav has consistent education and education_L values.


In [ ]:
EDUCATION_1990_MAP = {
    1.0: 'podstawowe nieukończone i bez wykształcenia',
    2.0: 'podstawowe',
    3.0: 'zasadnicze zawodowe',
    4.0: 'średnie',
    5.0: 'wyższe'
}
EDUCATION_9_MAP = {
    1.0: [1.0],
    2.0: [2.0],
    3.0: [3.0],
    4.0: [4.0, 5.0, 6.0, 7.0, 8.0],
    5.0: [9.0],
    np.nan: [98.0]
}
EDUCATION_1990_11_1_MAP = {
    1.0: [1.0, 2.0],
    2.0: [3.0, 4.0],
    3.0: [5.0],
    4.0: [6.0, 7.0, 8.0, 9.0, 10.0],
    5.0: [11.0],
    np.nan: [98.0]
}
EDUCATION_1990_11_2_MAP = {
    1.0: [1.0],
    2.0: [2.0, 3.0],
    3.0: [4.0],
    4.0: [5.0, 6.0, 7.0, 8.0, 9.0],
    5.0: [10.0, 11.0]
}
EDUCATION_1990_12_1_MAP = {
    1.0: [1.0],
    2.0: [2.0, 3.0],
    3.0: [4.0],
    4.0: [5.0, 6.0, 7.0, 8.0, 9.0],
    5.0: [10.0, 11.0]
}
EDUCATION_1990_12_2_MAP = {
    1.0: [1.0],
    2.0: [2.0, 3.0],
    3.0: [4.0],
    4.0: [5.0, 6.0, 7.0, 8.0],
    5.0: [9.0, 10.0, 11.0]
}

In [ ]:
# Drop rows with missing values in key variables
df = df[~df['sex'].isna()]
df = df[~df['age'].isna()]
df = df[~df['location_old'].isna()]
df = df[~df['location_new'].isna()]